<a href="https://colab.research.google.com/github/S00278393/secondrepo/blob/main/1_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: ECG Data Preparation with Apache Spark

This notebook performs data preparation for ECG classification using **real PTB-XL ECG data** and Apache Spark DataFrames.

**Dataset:** PTB-XL — a large publicly available clinical 12-lead ECG dataset  
https://physionet.org/content/ptb-xl/1.0.3/

**Steps:**
1. Environment setup (Java, Spark, PySpark, wfdb)
2. Download PTB-XL dataset metadata from PhysioNet
3. Map SCP codes to diagnostic superclasses and take a stratified sample
4. Load real ECG signal files and extract features
5. Load into Spark DataFrame and perform data cleaning
6. Feature engineering using Spark DataFrame operations
7. Analytical queries (filtering, grouping, aggregation)
8. Normalisation for ML readiness
9. Save processed data as CSV

## 1. Environment Setup

In [ ]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null 2>&1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# Install Java, Apache Spark, and processing libraries
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar -xzf spark-3.5.1-bin-hadoop3.tgz


In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PATH"] += ":/content/spark-3.5.1-bin-hadoop3/bin"

In [ ]:
!pip install -q wfdb

In [ ]:
!pip install --force-reinstall \
  numpy==1.26.4 \
  pandas==2.2.3 \
  pyarrow==14.0.2 \
  pyspark==3.5.1

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached pyarrow-14.0.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached pyspark-3.5.1-py2.py3-none-any.whl
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached py4j-0.10.9.7-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached pyarrow-14.0.2-cp312-cp312-manylinux_2_28_x86_64.whl (38.0 MB)
Using cached py4j-0.10.9.7-py2.py3-n

In [ ]:
!pip uninstall dataproc-spark-connect -y

In [ ]:
!pip uninstall bigframes datasets -y

## 2. Initialize Spark and Define Schema

In [ ]:
import json
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType,
    StringType, ArrayType
)

def create_spark_session(app_name="ECG_Preprocessing"):
    return (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.ui.port", "4050")
        .getOrCreate()
    )

def get_ecg_schema():
    return StructType([
        StructField("ecg_id", IntegerType(), False),
        StructField("patient_id", IntegerType(), False),
        StructField("age", DoubleType(), True),
        StructField("sex", IntegerType(), True),
        StructField("recording_date", StringType(), True),
        StructField("scp_codes", StringType(), True),
        StructField("diagnostic_class", StringType(), True),
        StructField("heart_rate", DoubleType(), True),
        StructField("signal_mean", DoubleType(), True),
        StructField("signal_std", DoubleType(), True),
        StructField("signal_max", DoubleType(), True),
        StructField("signal_min", DoubleType(), True),
        StructField("rr_interval_mean", DoubleType(), True),
        StructField("qrs_duration", DoubleType(), True),
        StructField("lead_I", ArrayType(DoubleType()), True),
        StructField("lead_II", ArrayType(DoubleType()), True),
    ])

spark = create_spark_session()
print("Spark session created successfully")

Spark session created successfully


## 3. Download and Load the PTB-XL Dataset

PTB-XL is a large publicly available 12-lead ECG dataset with 21,837 clinical recordings  
from 18,885 patients, annotated with 5 diagnostic superclasses:  
**NORM** (Normal), **MI** (Myocardial Infarction), **STTC** (ST/T-wave change),  
**CD** (Conduction Disturbance), **HYP** (Hypertrophy).

Reference: Wagner *et al.*, "PTB-XL, a large publicly available electrocardiography dataset,"  
*Scientific Data* 7, 154 (2020). https://physionet.org/content/ptb-xl/1.0.3/

In [ ]:
import ast
import os
import numpy as np
import pandas as pd
import wfdb

PTB_XL_DIR = '/content/ptb-xl'
os.makedirs(PTB_XL_DIR, exist_ok=True)

# Download PTB-XL metadata files from PhysioNet
print('Downloading PTB-XL metadata ...')
!wget -q -O /content/ptb-xl/ptbxl_database.csv \
    https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
!wget -q -O /content/ptb-xl/scp_statements.csv \
    https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv

df_meta = pd.read_csv(f'{PTB_XL_DIR}/ptbxl_database.csv', index_col='ecg_id')
scp_df  = pd.read_csv(f'{PTB_XL_DIR}/scp_statements.csv', index_col=0)

print(f'PTB-XL: {len(df_meta):,} total records')
print(f'Metadata columns: {list(df_meta.columns)}')
df_meta.head(3)

PTB-XL: 21,799 total records
Metadata columns: ['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']


,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr


In [ ]:
# Map SCP codes to the 5 diagnostic superclasses
SUPERCLASSES = {'NORM', 'MI', 'STTC', 'CD', 'HYP'}

def get_superclass(scp_codes_str):
    """
    Parse the SCP codes dict-string and return the first recognised
    diagnostic superclass (NORM / MI / STTC / CD / HYP).
    """
    try:
        scp_codes = ast.literal_eval(scp_codes_str)
    except (ValueError, SyntaxError):
        return None
    for code in scp_codes:
        if code in scp_df.index:
            dclass = scp_df.loc[code, 'diagnostic_class']
            if pd.notna(dclass) and dclass in SUPERCLASSES:
                return dclass
    return None

df_meta['diagnostic_class'] = df_meta['scp_codes'].apply(get_superclass)
df_labeled = df_meta[df_meta['diagnostic_class'].notna()].copy()

print(f'Records with a known diagnostic superclass: {len(df_labeled):,}')
print()
print('Diagnostic class distribution:')
print(df_labeled['diagnostic_class'].value_counts())

Records with a known diagnostic superclass: 21,388

Diagnostic class distribution:
diagnostic_class
NORM    9514
MI      5424
STTC    2817
CD      2325
HYP     1308
Name: count, dtype: int64


In [ ]:
RECORDS_PER_CLASS = 200

df_sample = (
    df_labeled
    .groupby('diagnostic_class', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), RECORDS_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)
print(f'Stratified sample: {len(df_sample)} records')
print(df_sample['diagnostic_class'].value_counts())


def extract_ecg_features(filename_lr, fs=100):
    """
    Read one PTB-XL 100 Hz record from PhysioNet via wfdb network interface
    and return a dict of signal-level features.

    Parameters
    ----------
    filename_lr : str  e.g. 'records100/00000/00001_lr'
    fs          : int  sampling frequency (100 Hz for low-res records)
    """
    try:
        folder = '/'.join(filename_lr.split('/')[:-1])
        name   = filename_lr.split('/')[-1]
        record = wfdb.rdrecord(name, pn_dir=f'ptb-xl/1.0.3/{folder}')
        sig     = record.p_signal              # shape: (1000, 12)
        lead_I  = sig[:, 0].astype(float)
        lead_II = sig[:, 1].astype(float)

        sig_mean = float(np.nanmean(lead_II))
        sig_std  = float(np.nanstd(lead_II))
        sig_max  = float(np.nanmax(lead_II))
        sig_min  = float(np.nanmin(lead_II))

        # Heart-rate via autocorrelation on Lead II
        # Search range: 0.33 s (180 bpm) to 2.0 s (30 bpm) — covers the
        # full physiologically plausible adult resting-heart-rate range.
        centered = lead_II - sig_mean
        acf = np.correlate(centered, centered, mode='full')
        acf = acf[len(acf) // 2:]
        lo = int(0.33 * fs)  # 180 bpm upper limit → 0.33 s minimum RR
        hi = int(2.0  * fs)  #  30 bpm lower limit → 2.00 s maximum RR
        if hi < len(acf):
            rr_samps    = lo + int(np.argmax(acf[lo:hi]))
            rr_interval = rr_samps / fs
            # Clamp to [30, 200] bpm: values outside this range are artefacts
            heart_rate  = float(np.clip(60.0 / rr_interval, 30, 200))
        else:
            rr_interval, heart_rate = 0.8, 75.0  # fallback defaults

        # QRS-duration estimate in ms via derivative threshold
        # The derivative of the ECG is large during rapid ventricular
        # depolarisation (QRS complex). We count samples where |d/dt| > 3σ
        # and convert to milliseconds.
        # Scaling factor 80: empirical calibration against known PTB-XL
        # QRS widths; result is clamped to the clinical range [60, 200] ms.
        deriv   = np.diff(centered)
        thr     = 3.0 * np.std(deriv)   # 3-sigma threshold
        n_qrs   = int(np.sum(np.abs(deriv) > thr))
        qrs_dur = float(np.clip(
            (n_qrs / max(len(deriv), 1)) * (1000 / fs) * 80,
            60,   # 60 ms: shortest normal QRS duration
            200   # 200 ms: threshold for extreme conduction delay
        ))

        return {
            'signal_mean'     : round(sig_mean, 6),
            'signal_std'      : round(sig_std, 6),
            'signal_max'      : round(sig_max, 6),
            'signal_min'      : round(sig_min, 6),
            'heart_rate'      : round(heart_rate, 2),
            'rr_interval_mean': round(rr_interval, 4),
            'qrs_duration'    : round(qrs_dur, 2),
            'lead_I'          : [float(x) for x in lead_I[:100]],
            'lead_II'         : [float(x) for x in lead_II[:100]],
        }
    except Exception:
        return None


print('Loading real ECG signals from PhysioNet'
      ' (this takes approx. 3-5 min for 1000 records) ...')
records_list = []
for idx, (ecg_id, row) in enumerate(df_sample.iterrows()):
    if idx % 100 == 0:
        print(f'  {idx}/{len(df_sample)} processed ...')
    feats = extract_ecg_features(row['filename_lr'])
    if feats is None:
        continue
    records_list.append({
        'ecg_id'           : int(ecg_id),
        'patient_id'       : int(row['patient_id']),
        'age'              : float(row['age'])  if pd.notna(row['age'])  else 50.0,
        'sex'              : int(row['sex'])    if pd.notna(row['sex'])  else 0,
        'recording_date'   : str(row['recording_date'])[:19],
        'scp_codes'        : str(row['scp_codes']),
        'diagnostic_class' : str(row['diagnostic_class']),
        **feats,
    })

print(f'Successfully loaded {len(records_list)} real ECG records')

/tmp/ipykernel_60599/1092159534.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), RECORDS_PER_CLASS), random_state=42))


Stratified sample: 1000 records
diagnostic_class
CD      200
HYP     200
MI      200
NORM    200
STTC    200
Name: count, dtype: int64
Loading real ECG signals from PhysioNet (this takes approx. 3-5 min for 1000 records) ...
  0/1000 processed ...
  100/1000 processed ...
  200/1000 processed ...
  300/1000 processed ...
  400/1000 processed ...
  500/1000 processed ...
  600/1000 processed ...
  700/1000 processed ...
  800/1000 processed ...
  900/1000 processed ...
Successfully loaded 1000 real ECG records


## 4. Load PTB-XL Data into Spark DataFrame

In [ ]:
# Convert real PTB-XL records to a Spark DataFrame
pdf    = pd.DataFrame(records_list)
ecg_df = spark.createDataFrame(pdf, schema=get_ecg_schema())

# =====================================================================
# BIG DATA SIMULATION: Force logical partitioning across Colab's CPU cores
# =====================================================================
original_partitions = ecg_df.rdd.getNumPartitions()

# Force the data into 50 chunks to simulate a massive distributed cluster
ecg_df = ecg_df.repartition(50)
distributed_partitions = ecg_df.rdd.getNumPartitions()

print("=== DISTRIBUTED COMPUTING PROOF ===")
print(f"Original Data Partitions: {original_partitions}")
print(f"Simulated Distributed Partitions: {distributed_partitions}")
print("==============================================")

print(f'\nDataFrame shape: {ecg_df.count()} rows, {len(ecg_df.columns)} columns')

print('\nExecution Plan (DAG):')
ecg_df.explain() # This outputs the physical Spark execution plan

print('\nSchema:')
ecg_df.printSchema()
ecg_df.show(5, truncate=True)

=== DISTRIBUTED COMPUTING PROOF FOR REPORT ===
Original Data Partitions: 2
Simulated Distributed Partitions: 50

DataFrame shape: 1000 rows, 16 columns

Execution Plan (DAG):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange RoundRobinPartitioning(50), REPARTITION_BY_NUM, [plan_id=15]
      +- *(1) Scan ExistingRDD[ecg_id#0,patient_id#1,age#2,sex#3,recording_date#4,scp_codes#5,diagnostic_class#6,heart_rate#7,signal_mean#8,signal_std#9,signal_max#10,signal_min#11,rr_interval_mean#12,qrs_duration#13,lead_I#14,lead_II#15]
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(50), REPARTITION_BY_NUM, [plan_id=10]
   +- Scan ExistingRDD[ecg_id#0,patient_id#1,age#2,sex#3,recording_date#4,scp_codes#5,diagnostic_class#6,heart_rate#7,signal_mean#8,signal_std#9,signal_max#10,signal_min#11,rr_interval_mean#12,qrs_duration#13,lead_I#14,lead_II#15]



Schema:
root
 |-- ecg_id: integer (nullable = false)
 |-- patient_id: integer (null

## 5. Data Cleaning and Transformation

In [ ]:
# Check for null values
print("--- Null Value Check ---")
null_counts = ecg_df.select(
    [F.sum(F.when(F.isnull(c), 1).otherwise(0)).alias(c) for c in ecg_df.columns]
)
null_counts.show(truncate=False)

# Remove any records with null heart_rate or diagnostic_class
ecg_clean = ecg_df.filter(
    F.col("heart_rate").isNotNull() & F.col("diagnostic_class").isNotNull()
)
print(f"Records after cleaning: {ecg_clean.count()}")

--- Null Value Check ---
+------+----------+---+---+--------------+---------+----------------+----------+-----------+----------+----------+----------+----------------+------------+------+-------+
|ecg_id|patient_id|age|sex|recording_date|scp_codes|diagnostic_class|heart_rate|signal_mean|signal_std|signal_max|signal_min|rr_interval_mean|qrs_duration|lead_I|lead_II|
+------+----------+---+---+--------------+---------+----------------+----------+-----------+----------+----------+----------+----------------+------------+------+-------+
|0     |0         |0  |0  |0             |0        |0               |0         |0          |0         |0         |0         |0               |0           |0     |0      |
+------+----------+---+---+--------------+---------+----------------+----------+-----------+----------+----------+----------+----------------+------------+------+-------+

Records after cleaning: 1000


In [ ]:
# Feature engineering: add derived columns
def add_derived_features(df):
    """Add clinically relevant derived features using Spark transformations."""
    return (
        df
        .withColumn("age_group",
            F.when(F.col("age") < 40, "Young")
            .when(F.col("age") < 60, "Middle-aged")
            .otherwise("Senior"))
        .withColumn("hr_category",
            F.when(F.col("heart_rate") < 60, "Bradycardia")
            .when(F.col("heart_rate") <= 100, "Normal")
            .otherwise("Tachycardia"))
        .withColumn("sex_label",
            F.when(F.col("sex") == 0, "Male").otherwise("Female"))
        .withColumn("signal_range",
            F.col("signal_max") - F.col("signal_min"))
        .withColumn("recording_year",
            F.year(F.to_timestamp("recording_date", "yyyy-MM-dd HH:mm:ss")))
    )

ecg_featured = add_derived_features(ecg_clean)
ecg_featured.select(
    "ecg_id", "age_group", "hr_category", "sex_label", "signal_range"
).show(5)

+------+-----------+-----------+---------+------------+
|ecg_id|  age_group|hr_category|sex_label|signal_range|
+------+-----------+-----------+---------+------------+
|   307|Middle-aged|Bradycardia|     Male|     -77.179|
|   355|     Senior|Bradycardia|   Female|     -42.314|
|   186|     Senior|Bradycardia|     Male|     -83.741|
|    76|      Young|Bradycardia|     Male|     -52.358|
|   388|     Senior|Bradycardia|   Female|     -68.392|
+------+-----------+-----------+---------+------------+
only showing top 5 rows



## 6. Analytical Queries Using Spark DataFrames

Demonstrate filtering, grouping, and aggregation functions.

In [ ]:
# Query 1: Diagnostic Class Distribution with Aggregation
print("=" * 60)
print("QUERY 1: Diagnostic Class Distribution")
print("=" * 60)

total = ecg_featured.count()
q1 = ecg_featured.groupBy("diagnostic_class").agg(
    F.count("*").alias("count"),
    F.round(F.count("*") / F.lit(total) * 100, 2).alias("percentage"),
    F.round(F.avg("heart_rate"), 2).alias("avg_heart_rate"),
    F.round(F.stddev("heart_rate"), 2).alias("std_heart_rate")
).orderBy(F.desc("count"))
q1.show()

QUERY 1: Diagnostic Class Distribution
+----------------+-----+----------+--------------+--------------+
|diagnostic_class|count|percentage|avg_heart_rate|std_heart_rate|
+----------------+-----+----------+--------------+--------------+
|              MI|  200|      20.0|           0.0|           0.0|
|             HYP|  200|      20.0|           0.0|           0.0|
|            NORM|  200|      20.0|           0.0|           0.0|
|            STTC|  200|      20.0|           0.0|           0.0|
|              CD|  200|      20.0|           0.0|          0.06|
+----------------+-----+----------+--------------+--------------+



In [ ]:
# Query 2: Heart Rate Statistics by Age Group and Sex (Grouping + Aggregation)
print("=" * 60)
print("QUERY 2: Heart Rate Stats by Age Group and Sex")
print("=" * 60)

q2 = ecg_featured.groupBy("age_group", "sex_label").agg(
    F.count("*").alias("count"),
    F.round(F.avg("heart_rate"), 2).alias("avg_hr"),
    F.round(F.min("heart_rate"), 2).alias("min_hr"),
    F.round(F.max("heart_rate"), 2).alias("max_hr"),
    F.round(F.stddev("heart_rate"), 2).alias("std_hr")
).orderBy("age_group", "sex_label")
q2.show()

QUERY 2: Heart Rate Stats by Age Group and Sex
+-----------+---------+-----+------+------+------+------+
|  age_group|sex_label|count|avg_hr|min_hr|max_hr|std_hr|
+-----------+---------+-----+------+------+------+------+
|Middle-aged|   Female|  117|   0.0| -0.01|  0.01|   0.0|
|Middle-aged|     Male|  181| -0.01| -0.78|  0.01|  0.06|
|     Senior|   Female|  308|   0.0| -0.02|  0.01|   0.0|
|     Senior|     Male|  312|   0.0| -0.01|  0.01|   0.0|
|      Young|   Female|   36|   0.0| -0.01|  0.01|   0.0|
|      Young|     Male|   46|   0.0| -0.01|  0.01|   0.0|
+-----------+---------+-----+------+------+------+------+



In [ ]:
# Query 3: Anomaly Detection using Filtering
print("=" * 60)
print("QUERY 3: Clinical Anomaly Detection")
print("=" * 60)

anomalies = ecg_featured.withColumn("clinical_flag",
    F.when((F.col("heart_rate") < 50) | (F.col("heart_rate") > 150), "Critical HR")
    .when(F.col("qrs_duration") > 140, "Wide QRS")
    .otherwise("Normal")
).filter(F.col("clinical_flag") != "Normal")

print(f"Total anomalous records: {anomalies.count()}")
anomalies.groupBy("clinical_flag", "diagnostic_class").count().orderBy(
    F.desc("count")
).show()

QUERY 3: Clinical Anomaly Detection
Total anomalous records: 1000
+-------------+----------------+-----+
|clinical_flag|diagnostic_class|count|
+-------------+----------------+-----+
|  Critical HR|              CD|  200|
|  Critical HR|             HYP|  200|
|  Critical HR|            NORM|  200|
|  Critical HR|            STTC|  200|
|  Critical HR|              MI|  200|
+-------------+----------------+-----+



In [ ]:
# Query 4: Temporal Trends (Grouping by Year)
print("=" * 60)
print("QUERY 4: Temporal Recording Trends")
print("=" * 60)

q4 = ecg_featured.groupBy("recording_year").agg(
    F.count("*").alias("total_recordings"),
    F.round(F.avg("heart_rate"), 2).alias("avg_hr"),
    F.countDistinct("diagnostic_class").alias("unique_diagnoses")
).orderBy("recording_year")
q4.show()

QUERY 4: Temporal Recording Trends
+--------------+----------------+------+----------------+
|recording_year|total_recordings|avg_hr|unique_diagnoses|
+--------------+----------------+------+----------------+
|          1984|               2|   0.0|               1|
|          1985|               1|   0.0|               1|
|          1986|              10|   0.0|               4|
|          1987|              42|   0.0|               5|
|          1988|              53|   0.0|               5|
|          1989|              54|   0.0|               5|
|          1990|             102| -0.01|               5|
|          1991|             104|   0.0|               5|
|          1992|              85|   0.0|               5|
|          1993|              98|   0.0|               5|
|          1994|              88|   0.0|               5|
|          1995|              77|   0.0|               5|
|          1996|              91|   0.0|               5|
|          1997|              68|   0

In [ ]:
# Query 5: Patient Risk Scoring with Window Functions
from pyspark.sql.window import Window

print("=" * 60)
print("QUERY 5: Patient Risk Scoring with Window Functions")
print("=" * 60)

scored_df = ecg_featured.withColumn("hr_risk",
    F.when(F.col("heart_rate") > 100, 1).otherwise(0)
).withColumn("age_risk",
    F.when(F.col("age") > 70, 1).otherwise(0)
).withColumn("risk_score", F.col("hr_risk") + F.col("age_risk"))

window_spec = Window.partitionBy("diagnostic_class").orderBy(F.desc("risk_score"))

ranked_df = (
    scored_df
    .withColumn("risk_rank", F.row_number().over(window_spec))
    .withColumn("class_avg_risk",
        F.round(F.avg("risk_score").over(
            Window.partitionBy("diagnostic_class")
        ), 2))
)

ranked_df.filter(F.col("risk_rank") <= 5).select(
    "ecg_id", "diagnostic_class", "risk_score", "risk_rank", "class_avg_risk"
).show(truncate=False)

QUERY 5: Patient Risk Scoring with Window Functions
+------+----------------+----------+---------+--------------+
|ecg_id|diagnostic_class|risk_score|risk_rank|class_avg_risk|
+------+----------------+----------+---------+--------------+
|65    |CD              |1         |1        |0.47          |
|143   |CD              |1         |2        |0.47          |
|70    |CD              |1         |3        |0.47          |
|150   |CD              |1         |4        |0.47          |
|2     |CD              |1         |5        |0.47          |
|341   |HYP             |1         |1        |0.38          |
|364   |HYP             |1         |2        |0.38          |
|250   |HYP             |1         |3        |0.38          |
|385   |HYP             |1         |4        |0.38          |
|242   |HYP             |1         |5        |0.38          |
|484   |MI              |1         |1        |0.36          |
|557   |MI              |1         |2        |0.36          |
|430   |MI        

In [ ]:
# Query 6: Cross-Tabulation of Diagnostic Class vs Heart Rate Category
print("=" * 60)
print("QUERY 6: Cross-Tabulation - Diagnosis vs HR Category")
print("=" * 60)

q6 = (
    ecg_featured.groupBy("diagnostic_class")
    .pivot("hr_category", ["Bradycardia", "Normal", "Tachycardia"])
    .agg(F.count("ecg_id"))
    .fillna(0)
    .orderBy("diagnostic_class")
)
q6.show(truncate=False)

# Row-wise percentages
total_per_class = (
    q6
    .withColumn("total",
        F.col("Bradycardia") + F.col("Normal") + F.col("Tachycardia"))
    .withColumn("brad_pct",
        F.round(F.col("Bradycardia") / F.col("total") * 100, 1))
    .withColumn("norm_pct",
        F.round(F.col("Normal") / F.col("total") * 100, 1))
    .withColumn("tachy_pct",
        F.round(F.col("Tachycardia") / F.col("total") * 100, 1))
    .select("diagnostic_class", "brad_pct", "norm_pct", "tachy_pct", "total")
)
total_per_class.show(truncate=False)

QUERY 6: Cross-Tabulation - Diagnosis vs HR Category
+----------------+-----------+------+-----------+
|diagnostic_class|Bradycardia|Normal|Tachycardia|
+----------------+-----------+------+-----------+
|CD              |200        |0     |0          |
|HYP             |200        |0     |0          |
|MI              |200        |0     |0          |
|NORM            |200        |0     |0          |
|STTC            |200        |0     |0          |
+----------------+-----------+------+-----------+

+----------------+--------+--------+---------+-----+
|diagnostic_class|brad_pct|norm_pct|tachy_pct|total|
+----------------+--------+--------+---------+-----+
|CD              |100.0   |0.0     |0.0      |200  |
|HYP             |100.0   |0.0     |0.0      |200  |
|MI              |100.0   |0.0     |0.0      |200  |
|NORM            |100.0   |0.0     |0.0      |200  |
|STTC            |100.0   |0.0     |0.0      |200  |
+----------------+--------+--------+---------+-----+



## 7. Normalization and ML-Ready Data Preparation

In [ ]:
from pyspark.ml.feature import StringIndexer, MinMaxScaler, VectorAssembler

# Select features for ML (drop raw signals and metadata)
# Raw leads (1000+ timesteps × 12 leads = 12,000+ features) cause the curse of dimensionality — sparse data,
# overfitting, and exponential sample requirements. Metadata like scp_codes (unstructured dict strings) and recording_date are non-numeric and
# incompatible with most ML algorithms (logistic regression, tree-based models).

ml_df = ecg_featured.select(
    "ecg_id", "age", "sex", "heart_rate", "signal_mean", "signal_std",
    "signal_max", "signal_min", "rr_interval_mean", "qrs_duration",
    "signal_range", "diagnostic_class"
)

# Encode the target label
indexer = StringIndexer(inputCol="diagnostic_class", outputCol="label")
ml_df = indexer.fit(ml_df).transform(ml_df)

# Assemble numeric features into a vector
numeric_cols = ["age", "sex", "heart_rate", "signal_mean", "signal_std",
                "signal_max", "signal_min", "rr_interval_mean",
                "qrs_duration", "signal_range"]

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features_raw")
ml_df = assembler.transform(ml_df)

# Normalize features using MinMaxScaler
scaler = MinMaxScaler(inputCol="features_raw", outputCol="features")
scaler_model = scaler.fit(ml_df)
ml_df = scaler_model.transform(ml_df)

print("ML-ready DataFrame schema:")
ml_df.select("ecg_id", "diagnostic_class", "label", "features").show(5, truncate=False)

ML-ready DataFrame schema:
+------+----------------+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ecg_id|diagnostic_class|label|features                                                                                                                                                            |
+------+----------------+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|334   |HYP             |1.0  |[0.20774647887323944,1.0,0.9828164665480064,0.3407254881773568,0.3649981731823165,0.9586622351474866,0.029867475440100228,0.8433734939759037,0.5,0.9718471105560326]|
|298   |HYP             |1.0  |[0.14788732394366197,0.0,0.98762232904708,0.1578379491403274,0.11198392400438438,0.9343581221437475,0.41933144326498323,0.18674698795180725,0.5,0.58684593

## 8. Save Processed Data as CSV

In [ ]:
# Save the flat (non-vector) columns as CSV for use in Notebook 2
csv_output = ecg_featured.select(
    "ecg_id", "patient_id", "age", "sex", "heart_rate", "signal_mean",
    "signal_std", "signal_max", "signal_min", "rr_interval_mean",
    "qrs_duration", "signal_range", "diagnostic_class",
    "age_group", "hr_category", "sex_label"
)

output_path = "/content/ecg_processed_data"
csv_output.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)

print(f"Data saved to {output_path}")
print(f"Total records saved: {csv_output.count()}")
csv_output.describe().show()

Data saved to /content/ecg_processed_data
Total records saved: 1000
+-------+-----------------+-----------------+------------------+-------------------+--------------------+-------------------+------------------+--------------------+-----------------+------------------+------------+-------------------+----------------+-----------+-----------+---------+
|summary|           ecg_id|       patient_id|               age|                sex|          heart_rate|        signal_mean|        signal_std|          signal_max|       signal_min|  rr_interval_mean|qrs_duration|       signal_range|diagnostic_class|  age_group|hr_category|sex_label|
+-------+-----------------+-----------------+------------------+-------------------+--------------------+-------------------+------------------+--------------------+-----------------+------------------+------------+-------------------+----------------+-----------+-----------+---------+
|  count|             1000|             1000|              1000|       

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change the output path to save directly to Drive
output_path = "/content/drive/MyDrive/ecg_processed_data"
csv_output.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)

Mounted at /content/drive


## Summary

This notebook completed:
- Downloaded the **real PTB-XL** metadata (21,837 records) from PhysioNet
- Mapped SCP codes to 5 diagnostic superclasses: NORM, MI, STTC, CD, HYP
- Took a stratified sample of up to 200 records per class (up to 1,000 total)
- Loaded actual ECG signal files via `wfdb` network interface and extracted features:
  - Signal statistics (mean, std, max, min of Lead II)
  - Heart rate (estimated via autocorrelation on Lead II)
  - QRS duration (derivative-threshold estimate in ms)
- Loaded real data into a Spark DataFrame and performed data cleaning
- Added derived features: age_group, hr_category, signal_range, recording_year, etc.
- Executed 6 analytical queries using Spark DataFrame operations:
  - Q1: Diagnostic distribution with aggregation
  - Q2: Heart-rate stats grouped by age and sex
  - Q3: Clinical anomaly detection via filtering
  - Q4: Temporal recording trends
  - Q5: Patient risk scoring with window functions
  - Q6: Cross-tabulation (Diagnosis vs Heart-rate category)
- Normalised features for machine learning
- Saved processed dataset as CSV for Notebook 2 (classification)